In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATH
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Dietary"
)

xpt_file = DATA_DIR / "DR2TOT_C.xpt"
csv_file = DATA_DIR / "DR2TOT_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 DR2TOT_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. BASIC STRUCTURE
# ============================================================

if len(df) == 9643:
    print("PASS: Row count = 9,643")
else:
    print("FAIL: Unexpected row count:", len(df))


if len(df.columns) == 81:
    print("PASS: Column count = 81")
else:
    print("FAIL: Unexpected column count:", len(df.columns))


# ============================================================
# 4. KNOWN PANDAS/XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


total_tiny = int(
    tiny_counts.sum()
)

tiny_variables = int(
    (tiny_counts > 0).sum()
)

total_true_zero = int(
    zero_counts.sum()
)


print("\n--- BEFORE CORRECTION ---")

print(
    "True numeric zeros:",
    f"{total_true_zero:,}"
)

print(
    "Tiny-value occurrences:",
    f"{total_tiny:,}"
)

print(
    "Variables containing tiny value:",
    tiny_variables
)


if total_tiny > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. VARIABLES VALIDATED TO ALLOW ZERO
# ============================================================

#
# These are the 68 fields in THIS DR2TOT_C XPT
# containing the tiny value.
#
# CDC permits 0 for these variables.
#
# DO NOT add another variable here unless validated.
#

ZERO_ALLOWED_COLUMNS = [

    "DR2TNUMF",

    "DR2TKCAL",
    "DR2TPROT",
    "DR2TCARB",
    "DR2TSUGR",
    "DR2TFIBE",
    "DR2TTFAT",
    "DR2TSFAT",
    "DR2TMFAT",
    "DR2TPFAT",
    "DR2TCHOL",
    "DR2TATOC",
    "DR2TATOA",
    "DR2TRET",
    "DR2TVARA",
    "DR2TACAR",
    "DR2TBCAR",
    "DR2TCRYP",
    "DR2TLYCO",
    "DR2TLZ",

    "DR2TVB1",
    "DR2TVB2",
    "DR2TNIAC",
    "DR2TVB6",
    "DR2TFOLA",
    "DR2TFA",
    "DR2TFF",
    "DR2TFDFE",
    "DR2TVB12",
    "DR2TB12A",
    "DR2TVC",
    "DR2TVK",

    "DR2TCALC",
    "DR2TPHOS",
    "DR2TMAGN",
    "DR2TIRON",
    "DR2TZINC",
    "DR2TCOPP",
    "DR2TSODI",
    "DR2TPOTA",
    "DR2TSELE",

    "DR2TCAFF",
    "DR2TTHEO",
    "DR2TALCO",
    "DR2TMOIS",

    "DR2TS040",
    "DR2TS060",
    "DR2TS080",
    "DR2TS100",
    "DR2TS120",
    "DR2TS140",
    "DR2TS160",
    "DR2TS180",

    "DR2TM161",
    "DR2TM181",
    "DR2TM201",
    "DR2TM221",

    "DR2TP182",
    "DR2TP183",
    "DR2TP184",
    "DR2TP204",
    "DR2TP205",
    "DR2TP225",
    "DR2TP226",

    "DR2_320",
    "DR2_330",
    "DR2BWATR",
    "DR2CWATR"
]


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_zero_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected_tiny = (
    actual_tiny_columns
    -
    approved_zero_columns
)


if unexpected_tiny:

    raise ValueError(
        "STOP: Tiny values appeared in variables "
        "that have NOT been approved for zero restoration:\n"
        + str(sorted(unexpected_tiny))
    )


missing_expected = (
    approved_zero_columns
    -
    set(df.columns)
)


if missing_expected:

    raise ValueError(
        "Expected variables are missing from this XPT:\n"
        + str(sorted(missing_expected))
    )


print(
    "\nPASS: Tiny values occur only in "
    "validated zero-permitted variables."
)


# ============================================================
# 7. RESTORE TRUE ZEROS
# ============================================================

correction_report = []


for col in ZERO_ALLOWED_COLUMNS:

    mask = df[col].eq(
        TINY_VALUE
    )

    count = int(
        mask.sum()
    )

    if count > 0:

        df.loc[
            mask,
            col
        ] = 0.0

        correction_report.append(
            {
                "variable": col,
                "restored_zeros": count
            }
        )


correction_df = pd.DataFrame(
    correction_report
)


print("\n--- ZERO RESTORATION ---")

print(
    correction_df
    .sort_values(
        "restored_zeros",
        ascending=False
    )
    .to_string(
        index=False
    )
)


print(
    "\nTotal zeros restored:",
    f"{correction_df['restored_zeros'].sum():,}"
)


# ============================================================
# 8. CLEAN SEQN ONLY AS IDENTIFIER
# ============================================================

if "SEQN" not in df.columns:

    raise ValueError(
        "SEQN not found."
    )


seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 9. DIETARY MEASUREMENTS
# ============================================================

#
# THESE MUST REMAIN AS-IS FROM XPT.
#
# No:
#
# .round()
# .astype(int)
# recalculation
# decimal truncation
# float_format
#

columns = df.columns.tolist()


nutrient_start = columns.index(
    "DR2TKCAL"
)

nutrient_end = columns.index(
    "DR2TP226"
)


DIETARY_MEASUREMENT_COLUMNS = [

    # Sample weights
    "WTDRD1",
    "WTDR2D",

    # Energy + all nutrient / fatty-acid variables
    *columns[
        nutrient_start:
        nutrient_end + 1
    ],

    # Water amounts
    "DR2_320",
    "DR2_330",
    "DR2BWATR",
    "DR2CWATR"
]


# ============================================================
# 10. BUILD EXPECTED XPT VALUES
# ============================================================

#
# Source is unchanged except validated:
#
# 5.397605346934028e-79 -> 0
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 11. VERIFY DIETARY MEASUREMENTS HAVE NOT CHANGED
# ============================================================

print("\n--- DIETARY MEASUREMENT FIDELITY ---")


for col in DIETARY_MEASUREMENT_COLUMNS:

    expected = (
        expected_df[col]
        .to_numpy()
    )

    current = (
        df[col]
        .to_numpy()
    )

    same = np.allclose(
        expected,
        current,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed."
        )


# ============================================================
# 12. CODE / COUNT VARIABLES
# ============================================================

#
# These are not dietary decimal measurements.
#
# They are codes/counts and can be represented
# without unnecessary .0.
#

CODE_COLUMNS = [

    "DR2DRSTZ",
    "DR2EXMER",
    "DRABF",
    "DRDINT",
    "DR2DAY",
    "DR2LANG",
    "DR2MNRSP",
    "DR2HELPD",

    "DR2TNUMF",

    "DR2_300",
    "DR2TWS"
]


print("\n--- CODE / COUNT CHECK ---")


for col in CODE_COLUMNS:

    values = (
        df[col]
        .dropna()
    )

    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values)
    )


    if fractional.any():

        raise ValueError(
            f"{col} contains unexpected fractional values. "
            f"Column was NOT converted."
        )


print(
    "PASS: Code/count variables contain "
    "whole-number source values."
)


# ============================================================
# 13. FORMAT CODE/COUNT VARIABLES
# ============================================================

for col in CODE_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


# ============================================================
# 14. CDC RECALL STATUS CHECK
# ============================================================

#
# CDC:
#
# 1 = Reliable             8220
# 2 = Not reliable           52
# 4 = Breast milk           134
# 5 = Not done             1237
#

EXPECTED_DR2DRSTZ = {

    1: 8220,
    2: 52,
    4: 134,
    5: 1237
}


actual_status = (
    df["DR2DRSTZ"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- DR2DRSTZ ---")

print(actual_status)


if actual_status == EXPECTED_DR2DRSTZ:

    print(
        "PASS: DR2DRSTZ matches CDC."
    )

else:

    print(
        "FAIL: DR2DRSTZ differs from CDC."
    )


# ============================================================
# 15. SAMPLE-WEIGHT CHECK
# ============================================================

print("\n--- SAMPLE WEIGHTS ---")


# WTDRD1

s = df["WTDRD1"]

print(
    "WTDRD1 valid:",
    int(s.notna().sum())
)

print(
    "WTDRD1 missing:",
    int(s.isna().sum())
)

print(
    "WTDRD1 min:",
    s.min()
)

print(
    "WTDRD1 max:",
    s.max()
)


if (
    int(s.notna().sum()) == 9034
    and
    int(s.isna().sum()) == 609
):

    print(
        "PASS: WTDRD1 counts match CDC."
    )

else:

    print(
        "FAIL: WTDRD1 counts differ."
    )


# WTDR2D

s = df["WTDR2D"]

print(
    "\nWTDR2D valid:",
    int(s.notna().sum())
)

print(
    "WTDR2D missing:",
    int(s.isna().sum())
)

print(
    "WTDR2D min:",
    s.min()
)

print(
    "WTDR2D max:",
    s.max()
)


if (
    int(s.notna().sum()) == 8354
    and
    int(s.isna().sum()) == 1289
):

    print(
        "PASS: WTDR2D counts match CDC."
    )

else:

    print(
        "FAIL: WTDR2D counts differ."
    )


# ============================================================
# 16. DR2TNUMF CHECK
# ============================================================

s = df["DR2TNUMF"]


print("\n--- DR2TNUMF ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if (
    int(s.notna().sum()) == 8354
    and
    int(s.isna().sum()) == 1289
    and
    s.min() == 0
    and
    s.max() == 42
):

    print(
        "PASS: DR2TNUMF matches CDC."
    )

else:

    print(
        "FAIL: DR2TNUMF differs from CDC."
    )


# ============================================================
# 17. ENERGY CHECK
# ============================================================

s = df["DR2TKCAL"]


print("\n--- DR2TKCAL ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)


if (
    int(s.notna().sum()) == 8220
    and
    int(s.isna().sum()) == 1423
    and
    s.min() == 0
    and
    s.max() == 9265
):

    print(
        "PASS: DR2TKCAL matches CDC."
    )

else:

    print(
        "FAIL: DR2TKCAL differs from CDC."
    )


# ============================================================
# 18. SELECTED DECIMAL NUTRIENT CHECKS
# ============================================================

print("\n--- DECIMAL NUTRIENT CHECKS ---")


# Protein

s = df["DR2TPROT"]

if (
    int(s.notna().sum()) == 8220
    and
    int(s.isna().sum()) == 1423
    and
    np.isclose(s.min(), 0)
    and
    np.isclose(s.max(), 407.31)
):

    print(
        "PASS: DR2TPROT matches CDC."
    )

else:

    print(
        "FAIL: DR2TPROT differs."
    )


# Carbohydrate

s = df["DR2TCARB"]

if (
    int(s.notna().sum()) == 8220
    and
    int(s.isna().sum()) == 1423
    and
    np.isclose(s.min(), 0)
    and
    np.isclose(s.max(), 1197.06)
):

    print(
        "PASS: DR2TCARB matches CDC."
    )

else:

    print(
        "FAIL: DR2TCARB differs."
    )


# ============================================================
# 19. WATER CHECKS
# ============================================================

print("\n--- WATER VARIABLES ---")


WATER_CHECKS = {

    "DR2_320": (
        0,
        11366.4,
        8364,
        1279
    ),

    "DR2_330": (
        0,
        11366.4,
        8374,
        1269
    ),

    "DR2BWATR": (
        0,
        7577.6,
        8394,
        1249
    ),

    "DR2CWATR": (
        0,
        2841.6,
        8406,
        1237
    )
}


for col, (
    expected_min,
    expected_max,
    expected_valid,
    expected_missing
) in WATER_CHECKS.items():

    s = df[col]

    passed = (

        int(s.notna().sum())
        ==
        expected_valid

        and

        int(s.isna().sum())
        ==
        expected_missing

        and

        np.isclose(
            s.min(),
            expected_min
        )

        and

        np.isclose(
            s.max(),
            expected_max
        )
    )


    if passed:

        print(
            f"PASS: {col:<10} "
            f"range={s.min()}-{s.max()}, "
            f"valid={int(s.notna().sum()):,}, "
            f"missing={int(s.isna().sum()):,}"
        )

    else:

        print(
            f"FAIL: {col}"
        )


# ============================================================
# 20. TAP WATER SOURCE CHECK
# ============================================================

EXPECTED_DR2TWS = {

    1: 4478,
    2: 424,
    3: 52,
    91: 30,
    99: 131
}


actual_tws = (
    df["DR2TWS"]
    .value_counts()
    .sort_index()
    .to_dict()
)


print("\n--- DR2TWS ---")

print(actual_tws)


if (
    actual_tws == EXPECTED_DR2TWS
    and
    int(df["DR2TWS"].isna().sum()) == 4528
):

    print(
        "PASS: DR2TWS matches CDC."
    )

else:

    print(
        "FAIL: DR2TWS differs from CDC."
    )


# ============================================================
# 21. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO FIDELITY ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny zero artifact remains."
    )

else:

    raise ValueError(
        "FAIL: Tiny values remain."
    )


# ============================================================
# 22. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format
# NO rounding
# NO recalculation
#
# Dietary values stay as read from XPT.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 23. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 24. EXACT DIETARY VALUE FIDELITY
# ============================================================

print("\n--- XPT -> CSV DIETARY FIDELITY ---")


measurement_failures = []


for col in DIETARY_MEASUREMENT_COLUMNS:

    expected = (
        expected_df[col]
        .to_numpy()
    )

    exported = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected,
        exported,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches XPT."
        )

    else:

        measurement_failures.append(
            col
        )

        print(
            f"FAIL: {col} changed."
        )


if not measurement_failures:

    print(
        "\nPASS: All dietary measurement "
        "values preserved exactly."
    )


# ============================================================
# 25. MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )


# ============================================================
# 26. RAW SEQN FORMAT CHECK
# ============================================================

print("\n--- RAW CSV SEQN CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    f.readline()

    bad_seqn = 0

    for line in f:

        seqn_text = (
            line.split(",", 1)[0]
        )

        if seqn_text.endswith(".0"):

            bad_seqn += 1


print(
    "SEQN values ending in .0:",
    bad_seqn
)


if bad_seqn == 0:

    print(
        "PASS: SEQN exported without .0."
    )


print("\n" + "=" * 75)
print("DR2TOT_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 DR2TOT_C XPT -> CSV
Rows: 9,643
Columns: 81
PASS: Row count = 9,643
PASS: Column count = 81

--- BEFORE CORRECTION ---
True numeric zeros: 0
Tiny-value occurrences: 69,290
Variables containing tiny value: 68

Tiny values by variable:
DR2CWATR    8203
DR2TALCO    7426
DR2TATOA    7379
DR2TP184    5982
DR2BWATR    5506
DR2TB12A    5231
DR2TTHEO    4441
DR2TP225    4254
DR2_330     3291
DR2TP205    2894
DR2TCAFF    2370
DR2TLYCO    2285
DR2TP226    2109
DR2TM221    1989
DR2_320     1351
DR2TACAR     897
DR2TCRYP     869
DR2TS040     520
DR2TS060     427
DR2TP204     411
DR2TS080     266
DR2TFA       155
DR2TM201     124
DR2TS100     120
DR2TBCAR     104
DR2TLZ        98
DR2TFIBE      82
DR2TFF        68
DR2TRET       61
DR2TCHOL      57
DR2TVC        42
DR2TM161      37
DR2TVB12      31
DR2TS120      22
DR2TVARA      18
DR2TVK        13
DR2TS140      11
DR2TS180       8
DR2TP183       8
DR2TVB6        8
DR2TATOC       8
DR2TFDFE       7
DR2TVB1        7
DR2TMFAT       7
D